In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from torch.distributions import Categorical
import torch.nn.functional as F
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

In [2]:
def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (84, 84))
    frame = frame / 255.0
    return frame

# Stack 4 frames together to create temporal awareness for the agent
def stack_frames(stacked_frames, new_frame, is_new_episode):
    if is_new_episode:
        stacked_frames = deque([np.zeros((84, 84), dtype=np.float32) for _ in range(4)], maxlen=4)
        for _ in range(4):
            stacked_frames.append(new_frame)
    else:
        stacked_frames.append(new_frame)

    stacked_state = np.stack(stacked_frames, axis=0)
    return stacked_state, stacked_frames

In [3]:
c = deque([np.zeros((84, 84), dtype=np.float32) for i in range(10)])
c[0].shape
stacked_state = np.stack(c, axis=0)
stacked_state.shape

(10, 84, 84)

In [4]:
class A2CNetwork(nn.Module):
    def __init__(self, input_channels, action_space):
        super(A2CNetwork, self).__init__()
        
        # Define the four convolutional layers
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1)  # New conv layer
        
        # Calculate the input size to the fully connected layer after conv4
        # Assuming the input image size is 84x84 as in many RL environments
        self.fc1 = nn.Linear(128 * 5 * 5, 512)  # Update the input size here
        
        # Actor and Critic heads
        self.actor = nn.Linear(512, action_space)
        self.critic = nn.Linear(512, 1)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))  # Pass through the new convolutional layer
        x = x.view(x.size(0), -1)  # Flatten the output for the fully connected layer
        x = F.relu(self.fc1(x))
        policy = self.actor(x)
        value = self.critic(x)
        return policy, value

    def select_action(self, x):
        logits, _ = self.forward(x)
        action_probs = F.softmax(logits, dim=1)
        action_probs = torch.clamp(action_probs, 1e-6, 1.0)  # Prevent sampling errors
        action = torch.multinomial(action_probs, 1).item()
        return action


In [5]:
class A2CAgent:

    def __init__(self, env, network, optimizer, gamma=0.99):
        self.env = env
        self.network = network
        self.optimizer = optimizer
        self.gamma = gamma
        self.stacked_frame = deque(maxlen=4)

    def preprocess_state(self, state, is_new_episode):
        processed_frame = preprocess_frame(state)
        stacked_state, self.stacked_frame = stack_frames(self.stacked_frame, processed_frame, is_new_episode)
        return stacked_state

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0)
        policy, value = self.network(state)
        probs = torch.softmax(policy, dim=-1)
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action), value

    def compute_returns(self, rewards):
        returns = []
        R = 0
        for r in reversed(rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        return returns

    def train(self, num_episodes=1000):
        for episode in range(num_episodes):
            state, _ = self.env.reset()

            # print(f'state getting out of env: {state.shape}')

            state = self.preprocess_state(state, is_new_episode=True)

            # print(f'state getting out of preprocess: {state.shape}')

            log_probs = []
            values = []
            rewards = []

            done = False
            while not done:
                action, log_prob, value = self.select_action(state)
                next_state, reward, done, _, _ = self.env.step(action)
                next_state = self.preprocess_state(next_state, is_new_episode=False)

                log_probs.append(log_prob)
                values.append(value)
                rewards.append(reward)

                state = next_state

            returns = self.compute_returns(rewards)
            policy_loss = []
            value_loss = []
            for log_prob, value, R in zip(log_probs, values, returns):
                advantage = R - value.item()
                policy_loss.append(-log_prob * advantage)
                value_loss.append(nn.functional.mse_loss(value, torch.tensor([R])))

            self.optimizer.zero_grad()
            # !!!!!!!!!!!!!!!!!!!!!!! why use stack?
            loss = torch.stack(policy_loss).sum() + torch.stack(value_loss).sum()
            loss.backward()
            self.optimizer.step()

            if episode % 10 == 0:
                print(f"Episode {episode}: Total Reward = {sum(rewards)}")

In [6]:
env = gym.make('ALE/SpaceInvaders-v5', render_mode='rgb_array')
network = A2CNetwork(4, env.action_space.n)
state_dict = torch.load('./checkpoints/a2c_model_checkpoint.pth', map_location='cpu')
network.load_state_dict(state_dict['model_state_dict'])
optimizer = optim.Adam(network.parameters(), lr=1e-3)
agent = A2CAgent(env, network, optimizer)

# agent.train(num_episodes=1000)

C:\Users\Amiroodi\AppData\Local\Temp\ipykernel_2820\1002682115.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('./checkpoints/a2c_model_checkpoin

In [7]:
def play_trained_agent(env, network, episodes=5):
    for episode in range(episodes):
        state, _ = env.reset()
        state = preprocess_frame(state)
        stacked_frames = deque([np.zeros((84, 84), dtype=np.float32) for _ in range(4)], maxlen=4)
        stacked_state, stacked_frames = stack_frames(stacked_frames, state, is_new_episode=True)

        done = False
        total_reward = 0

        while not done:
            env.render()
            state_tensor = torch.FloatTensor(stacked_state).unsqueeze(0)
            policy, value = network(state_tensor)
            print(policy.shape)
            print(policy.squeeze(0))
            action = torch.argmax(policy.squeeze(0), dim=0).item()

            next_state, reward, done, _, _ = env.step(action)
            next_state = preprocess_frame(next_state)
            stacked_state, stacked_frames = stack_frames(stacked_frames, state, is_new_episode=False)

            total_reward += reward

        print(f"Episode {episode + 1} ended with total reward: {total_reward}")

In [8]:
env = gym.make("ALE/SpaceInvaders-v5", render_mode="human")
play_trained_agent(env, network)

torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -0.0167,  0.0269],
       grad_fn=<SqueezeBackward1>)
torch.Size([1, 6])
tensor([ 0.0237, -0.0041, -0.0162, -0.0029, -

: 

In [ ]:
s = [1., 2., 3.]
s = torch.tensor(s)
print(s)
s.sum()

tensor([1., 2., 3.])


tensor(6.)

In [ ]:
a = [1, 2, 3]
b = ['a', 'b', 'c']
for i, j in zip(a, b):
    print(i, j)

1 a
2 b
3 c


In [ ]:
returns = [1, 2]
returns.insert(0, 4)
returns

[4, 1, 2]

In [ ]:
policy = torch.tensor([1., 6., 8., 18.])
probs = torch.softmax(policy, dim=-1)
m = Categorical(probs)
print(probs)
action = m.sample()
m.log_prob(action)

tensor([4.1397e-08, 6.1439e-06, 4.5398e-05, 9.9995e-01])


tensor(-5.1559e-05)

In [ ]:
a = np.random.rand(4, 84, 84)
a = torch.from_numpy(a)
a = a.unsqueeze(0)
a.shape

torch.Size([1, 4, 84, 84])

In [ ]:
a = np.random.rand(4, 84, 84)
a = np.float32(a)
a = torch.from_numpy(a)
net = A2CNetwork(4, 2)
b = net(a)
b.shape

AttributeError: 'tuple' object has no attribute 'shape'

: 

In [ ]:
b.view(64, -1).shape

torch.Size([64, 49])

In [ ]:
a = np.array([1, 2])
np.float32(a)

array([1., 2.], dtype=float32)

In [ ]:
np.stack([[1, 2], [2, 3]], axis=0)

array([[1, 2],
       [2, 3]])